In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()


In [0]:
# =============================================================================
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle (desafio_kinea.research.controle_fontes) a cada execução.
# =============================================================================
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"


In [0]:
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests


In [0]:
try:
    dbutils.widgets.remove("fonte")
except Exception:
    pass

dbutils.widgets.text("fonte", "todas")
NOME_FONTE = dbutils.widgets.get("fonte")

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

PASTA_BASE = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}"

PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]
HTTP_TIMEOUT = 30
MIN_CHARS_TEXTO = 200

TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form",
             "nav", "header", "footer", "aside", "button"]
SELETORES_CONTEUDO = ["#content-core", "#parent-fieldname-text", "#content", "main",
                       ".field--name-body", ".single_meta .blog_content", "#page-document",
                       ".entry-content", ".elementor-widget-theme-post-content", ".td-post-content",
                       ".artigo__texto"]  # AGERGS (CMS Matriz)


In [0]:
def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception:
        return set()


def salvar_manifesto(caminho: str, urls: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(urls), f, ensure_ascii=False, indent=2)


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=url)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    base = None
    for seletor in SELETORES_CONTEUDO:
        encontrado = soup.select_one(seletor)
        if encontrado and len(encontrado.get_text(strip=True)) > 300:
            base = encontrado
            break
    if base is None:
        article = soup.find("article")
        base = article if (article and len(article.get_text(strip=True)) > 500) else soup

    texto = base.get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", texto).strip()


def extrair_titulo_h1(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    if h1:
        texto = h1.get_text(" ", strip=True)
        return texto or None
    return None


In [0]:
# --- Acende Brasil ---

PADRAO_URL_ACENDE = re.compile(r"/artigo/[a-z0-9\-]+/?$", re.IGNORECASE)
PADRAO_DATA_ACENDE = re.compile(r"Data da publica[çc][ãa]o:\s*(\d{2}/\d{2}/\d{4})")


def listar_acende_brasil(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()
    for tag_a in soup.find_all("a", href=True):
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if not PADRAO_URL_ACENDE.search(url_absoluta) or url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)
        itens.append({"titulo": tag_a.get_text(" ", strip=True), "url": url_absoluta})
    return itens


def data_acende_brasil(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_ACENDE.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"


# --- ANTT ---

COLECAO_BASE_ANTT = "https://www.gov.br/antt/pt-br/assuntos/noticias-defeso-eleitoral"
PADRAO_DATA_ANTT = re.compile(r"Publicado em\s*(\d{2}/\d{2}/\d{4})")


def listar_antt(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()
    for tag_a in soup.find_all("a", href=True):
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if not url_absoluta.startswith(COLECAO_BASE_ANTT + "/") or "?" in url_absoluta:
            continue
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)
        titulo = tag_a.get_text(" ", strip=True)
        if titulo:
            itens.append({"titulo": titulo, "url": url_absoluta})
    return itens


def data_antt(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_ANTT.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"


# --- Agesan-RS Notícias ---

PADRAO_DATA_AGESAN = re.compile(r"(\d{1,2}/\d{2}/\d{2})\s")
TITULOS_EXCLUIR_AGESAN = ["extrato de aviso prévio"]


def listar_agesan_noticias(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()
    for h2 in soup.find_all("h2"):
        tag_a = h2.find("a", href=True)
        if not tag_a:
            continue
        titulo = tag_a.get_text(" ", strip=True)
        if not titulo or any(t in titulo.lower() for t in TITULOS_EXCLUIR_AGESAN):
            continue
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)

        data_publicacao = None
        proximo = h2.find_next(string=PADRAO_DATA_AGESAN)
        if proximo:
            m = PADRAO_DATA_AGESAN.search(proximo)
            if m:
                dia, mes, ano = m.group(1).split("/")
                ano_completo = f"20{ano}" if len(ano) == 2 else ano
                data_publicacao = f"{ano_completo}-{mes}-{dia.zfill(2)}"

        itens.append({"titulo": titulo, "url": url_absoluta, "published_at": data_publicacao})
    return itens


# --- PSR (via Exame) ---

MESES_PT = {
    "janeiro": "01", "fevereiro": "02", "março": "03", "abril": "04",
    "maio": "05", "junho": "06", "julho": "07", "agosto": "08",
    "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12",
}
PADRAO_DATA_EXAME = re.compile(r"(\d{1,2}) de (\w+) de (\d{4})")

CAMINHOS_EXCLUIR_EXAME = [
    "/politica-de-privacidade/", "/termos-de-uso/", "/politica-de-cookies/",
    "/lei-transparencia-salarial/", "/institucional/", "/newsletters/",
    "/canais-especiais/", "/faculdade", "/revista-exame/",
    "/invest/calculadoras/", "/invest/guia/",
    "materias-em-destaque",
]


def listar_psr_exame(html: str, url_base: str, max_paginas: int = 2) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_base if pagina == 1 else f"{url_base.rstrip('/')}/{pagina}/"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        for tag_a in soup.find_all("a", href=True):
            href = tag_a["href"].strip()
            if "/arquivo/" in href or href in vistos:
                continue
            if not href.startswith("https://exame.com/") or href.rstrip("/") == "https://exame.com":
                continue
            if any(caminho in href for caminho in CAMINHOS_EXCLUIR_EXAME):
                continue
            titulo = tag_a.get_text(" ", strip=True)
            if len(titulo) < 20:
                continue
            vistos.add(href)
            itens.append({"titulo": titulo, "url": href})

    return itens


def data_psr_exame(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_EXAME.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes_nome, ano = m.groups()
    mes = MESES_PT.get(mes_nome.lower())
    if not mes:
        return None
    return f"{ano}-{mes}-{dia.zfill(2)}"


def extrair_titulo_h1_exame(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    for h1 in soup.find_all("h1"):
        texto = h1.get_text(" ", strip=True)
        if texto and len(texto) > 15 and texto.lower() not in ("esg", "economia", "negócios"):
            return texto
    return None


# --- ANEEL (mesma plataforma Plone do ANTT, generalizada) ---

PADRAO_DATA_GOVBR = re.compile(r"Publicado em\s*(\d{2}/\d{2}/\d{4})")


def listar_govbr_plone(html: str, url_base: str, colecao_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()
    for tag_a in soup.find_all("a", href=True):
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if not url_absoluta.startswith(colecao_base + "/") or "?" in url_absoluta:
            continue
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)
        titulo = tag_a.get_text(" ", strip=True)
        if titulo:
            itens.append({"titulo": titulo, "url": url_absoluta})
    return itens


def data_govbr_plone(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_GOVBR.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"

# --- Agetransp ---

PADRAO_DATA_LISTAGEM_AGETRANSP = re.compile(r"(\d{2}/\d{2}/\d{4})")


def listar_agetransp(html: str, url_base: str) -> list[dict]:
    itens, vistos = [], set()

    for tag_a in BeautifulSoup(html, "lxml").find_all("a", href=True):
        href = tag_a["href"].strip()
        url_absoluta = urllib.parse.urljoin(url_base, href)

        if not url_absoluta.startswith("https://www.agetransp.rj.gov.br/noticias/"):
            continue
        if url_absoluta in vistos:
            continue

        titulo = tag_a.get_text(" ", strip=True)
        if len(titulo) < 15:
            continue
        vistos.add(url_absoluta)

        data_publicacao = None
        proximo_texto = tag_a.find_next(string=PADRAO_DATA_LISTAGEM_AGETRANSP)
        if proximo_texto:
            m = PADRAO_DATA_LISTAGEM_AGETRANSP.search(proximo_texto)
            if m:
                dia, mes, ano = m.group(1).split("/")
                data_publicacao = f"{ano}-{mes}-{dia}"

        itens.append({"titulo": titulo, "url": url_absoluta, "published_at": data_publicacao})

    return itens


# --- ANA (Notícias, período eleitoral 2026) ---
#
# TODO (pós-período eleitoral 2026): "site_url" desta fonte, na entrada
# "ana" de CONFIGS_FONTES logo abaixo, é temporária -- a própria página da
# ANA avisa que esse endereço só existe por causa da legislação eleitoral
# de 2026. Depois do período eleitoral a ANA volta a publicar no endereço
# histórico de notícias (ainda não identificado) -- atualizar o "site_url"
# quando isso acontecer. O resto (listar_ana, extrair_titulo_h1,
# data_govbr_plone) deve continuar funcionando sem mudança: mesma
# plataforma Plone/gov.br de ANTT e ANEEL.
#
# Sem filtro de relevância aqui de propósito -- a listagem real mistura
# ruído administrativo (ex.: lista telefônica, prêmios) com conteúdo
# regulatório relevante (consultas públicas, revisão tarifária, agenda
# regulatória); filtrar por relevância é responsabilidade da etapa de NLP
# mais adiante no pipeline, não da captura.

ITENS_POR_PAGINA_ANA = 30


def listar_ana(html: str, url_base: str, max_paginas: int = 5) -> list[dict]:
    itens = []

    for pagina in range(max_paginas):
        b_start = pagina * ITENS_POR_PAGINA_ANA
        html_pagina = html if pagina == 0 else baixar_pagina(f"{url_base.rstrip('/')}?b_start:int={b_start}")
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        lista = soup.select_one("ul.noticias.listagem-noticias-com-foto")
        if not lista:
            break

        lis = lista.find_all("li", recursive=False)
        if not lis:
            break

        for li in lis:
            tag_a = li.select_one("h2.titulo a")
            if not tag_a:
                continue

            data_publicacao = None
            tag_data = li.select_one("span.data")
            if tag_data:
                m = re.match(r"(\d{2})/(\d{2})/(\d{4})", tag_data.get_text(strip=True))
                if m:
                    dia, mes, ano = m.groups()
                    data_publicacao = f"{ano}-{mes}-{dia}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": tag_a["href"].strip(),
                "published_at": data_publicacao,
            })

        if len(lis) < ITENS_POR_PAGINA_ANA:
            break

    return itens


# --- AGENERSA (RJ) — Notícias ---
#
# Listagem é uma View padrão do Drupal (`/agenersa/noticias`, paginação
# `?page=N`, 0-indexed). A data de publicação já vem pronta na própria
# listagem, num `<time datetime="...">` -- não precisa abrir a página de
# detalhe só para descobrir a data (por isso "extrair_data": None no
# CONFIGS_FONTES abaixo, igual a agesan_noticias/agetransp).
#
# Testado isoladamente em ingestores/SANEAMENTO/Agências_Reguladoras_Estaduais_RJ.ipynb
# (Fase 1): robots.txt do domínio atual (www.rj.gov.br) NÃO bloqueia esta
# listagem nem as páginas de notícia -- diferente do que constava
# anteriormente em controle_fontes (bloqueio provavelmente registrado
# contra o domínio antigo, agenersa.rj.gov.br, hoje um 301 para cá).


def listar_agenersa(html: str, url_base: str, max_paginas: int = 5) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(max_paginas):
        html_pagina = html if pagina == 0 else baixar_pagina(f"{url_base.rstrip('/')}?page={pagina}")
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        linhas = soup.select("div.views-row")
        if not linhas:
            break

        for linha in linhas:
            tag_a = linha.select_one("h2.field-content a[href]")
            if not tag_a:
                continue
            url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
            if url_absoluta in vistos:
                continue
            vistos.add(url_absoluta)

            data_publicacao = None
            tag_time = linha.select_one("time[datetime]")
            if tag_time and tag_time.get("datetime"):
                data_publicacao = tag_time["datetime"][:10]

            itens.append({
                "titulo": tag_a.get_text(" ", strip=True),
                "url": url_absoluta,
                "published_at": data_publicacao,
            })

    return itens


# --- ABEGÁS — Notícias do Setor ---
#
# WordPress com page builder (Elementor + tema GT3, layout packery/
# isotope) -- não é o loop padrão de WP, os seletores são do tema:
# listagem em div.blog_post_preview.noticias, paginação por path
# (/noticias-do-setor/page/N, não querystring). Data já vem pronta na
# listagem (span.post_date) -- "extrair_data": None no CONFIGS_FONTES,
# mesmo padrão de agesan_noticias/agenersa. O seletor de conteúdo
# (.single_meta .blog_content) foi acrescentado a SELETORES_CONTEUDO
# (célula de configuração) em vez de virar uma função própria, já que
# extrair_texto_generico() já cobre esse caso.
#
# API REST do WP revelou 14.274 posts na categoria "Notícias" -- histórico
# enorme, por isso max_paginas=5 (50 itens mais recentes por execução) em
# vez de tentar cobrir o arquivo inteiro.

ITENS_POR_PAGINA_ABEGAS = 10


def listar_abegas(html: str, url_base: str, max_paginas: int = 5) -> list[dict]:
    itens = []

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_base if pagina == 1 else f"{url_base.rstrip('/')}/page/{pagina}"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        itens_pagina = soup.select("div.blog_post_preview.noticias")
        if not itens_pagina:
            break

        for item in itens_pagina:
            tag_a = item.select_one("h2.blogpost_title a")
            if not tag_a:
                continue

            data_publicacao = None
            tag_data = item.select_one("span.post_date")
            if tag_data:
                m = re.match(r"(\d{2})/(\d{2})/(\d{4})", tag_data.get_text(strip=True))
                if m:
                    dia, mes, ano = m.groups()
                    data_publicacao = f"{ano}-{mes}-{dia}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": tag_a["href"].strip(),
                "published_at": data_publicacao,
            })

        if len(itens_pagina) < ITENS_POR_PAGINA_ABEGAS:
            break

    return itens


# --- ANATEL — Notícias ---
#
# Plone 6 / Volto (frontend React, conteúdo em blocos) -- arquitetura
# diferente de ANA/ANP (Plone clássico), apesar de ser a mesma família
# gov.br. A LISTAGEM usa a API REST pública do Plone (++api++/@search,
# sem autenticação) porque o HTML cru da página de busca é só a casca do
# SearchBlock (React), sem nenhum item -- mesma situação da ONS.
#
# Diferente da ONS, porém, cada NOTÍCIA individual (não a listagem) é
# server-side renderizada com o texto completo já no HTML -- por isso
# reaproveita extrair_texto_generico() normalmente (seletor "#page-document"
# acrescentado a SELETORES_CONTEUDO), sem precisar de extrator próprio
# nem de segunda chamada à API por item.
#
# "site_url" no CONFIGS_FONTES já é a URL completa da API de busca (não a
# página pública) -- baixar_pagina() não faz distinção, só baixa texto;
# quem interpreta como JSON é o listar_anatel() abaixo. O "path" do filtro
# tem que ser relativo à raiz do site ("/pt-br/assuntos/noticias"), sem o
# prefixo "/anatel/" -- com o prefixo o filtro devolve 0 resultados sem
# erro nenhum (silencioso).
#
# Histórico pequeno (16 itens na pasta) -- sem paginação necessária,
# b_size grande cobre tudo numa chamada só.


def listar_anatel(html: str, url_base: str) -> list[dict]:
    try:
        dados = json.loads(html)
    except Exception:
        return []

    itens = []
    for item in dados.get("items", []):
        efetiva = item.get("effective")
        itens.append({
            "titulo": item.get("title"),
            "url": item.get("@id"),
            "published_at": efetiva[:10] if efetiva else None,
        })
    return itens


# --- ABAR — Acontece nas Agências ---
#
# Agregador nacional (notícias de 87 agências reguladoras associadas --
# saneamento, energia, transporte, recursos hídricos, estaduais e
# municipais). WordPress com tema magazine (família "Jannah"/jeg) --
# listagem em article.jeg_post, paginação por path (/page/N, igual
# ABEGÁS). Sem filtro de relevância aqui de propósito (mesmo raciocínio
# de ANA/ANP): a etapa de NLP mais adiante decide o que é relevante.
#
# Armadilha do tema: cada página da categoria mostra um bloco "hero" fixo
# (os 4 posts mais recentes, <h2 class="jeg_post_title">) seguido da lista
# de fato paginada (<h3 class="jeg_post_title">) -- ambos com a mesma
# classe article.jeg_post no elemento pai. Um seletor restrito a "h2" só
# pega o hero e (por ele ser fixo) parece que a paginação não funciona;
# seletor certo é ".jeg_post_title a" (sem restringir a tag), com dedup
# por URL entre páginas para não repetir os itens do hero. Data já vem
# pronta na listagem em formato longo ("31 de julho de 2026"), mesmo
# padrão de data do PSR/Exame (reaproveita MESES_PT). Histórico grande
# (~1.042 posts na categoria) -- max_paginas=5 cobre só os mais recentes
# por execução, mesmo critério de ANP/ABEGÁS.

PADRAO_DATA_ABAR = re.compile(r"(\d{1,2}) de (\w+) de (\d{4})")
ITENS_POR_PAGINA_ABAR = 14


def listar_abar(html: str, url_base: str, max_paginas: int = 5) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_base if pagina == 1 else f"{url_base.rstrip('/')}/page/{pagina}/"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        itens_pagina = soup.select("article.jeg_post")
        if not itens_pagina:
            break

        for item in itens_pagina:
            tag_a = item.select_one(".jeg_post_title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)

            data_publicacao = None
            tag_data = item.select_one(".jeg_meta_date")
            if tag_data:
                m = PADRAO_DATA_ABAR.search(tag_data.get_text(" ", strip=True))
                if m:
                    dia, mes_nome, ano = m.groups()
                    mes = MESES_PT.get(mes_nome.lower())
                    if mes:
                        data_publicacao = f"{ano}-{mes}-{dia.zfill(2)}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": url_item,
                "published_at": data_publicacao,
            })

        if len(itens_pagina) < ITENS_POR_PAGINA_ABAR:
            break

    return itens

# --- EPE ---

PADRAO_DATA_EPE = re.compile(r"(\d{2}/\d{2}/\d{4})")


def listar_epe(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens, vistos = [], set()

    for item_div in soup.find_all("div", class_="item"):
        tag_a = item_div.find("a", href=True)
        if not tag_a:
            continue

        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        if "/area-" in url_absoluta or url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)

        h2 = tag_a.find("h2")
        titulo = h2.get_text(" ", strip=True) if h2 else tag_a.get_text(" ", strip=True)
        if not titulo:
            continue

        data_publicacao = None
        span_data = item_div.find("span", class_="date")
        if span_data:
            m = PADRAO_DATA_EPE.search(span_data.get_text())
            if m:
                dia, mes, ano = m.group(1).split("/")
                data_publicacao = f"{ano}-{mes}-{dia}"

        itens.append({"titulo": titulo, "url": url_absoluta, "published_at": data_publicacao})

    return itens

# --- Instituto Trata Brasil (Blog) ---
#
# WordPress com Elementor -- loop de posts em div.e-loop-item (não é o
# article.post classico nem um page builder de terceiros como GT3/jeg).
# Titulo/link em h1.elementor-heading-title a, categoria em
# .elementor-post-info__terms-list-item, data em
# [itemprop="datePublished"] time (DD/MM/YYYY), resumo (nao usado no
# metadados, so pra conferencia) em .elementor-widget-theme-post-excerpt.
# Texto completo do post em .elementor-widget-theme-post-content
# (acrescentado a SELETORES_CONTEUDO -- esse tema nao usa .entry-content).
# Paginacao /blog/N/, ~300 posts / 30 paginas no total -- max_paginas
# conservador, mesmo criterio de ANP/ABEGAS/ABAR.
#
# Conteudo mais analitico/institucional (estudos, rankings, dados de
# investimento) do que noticia factual pontual de agencia reguladora --
# sem filtro de relevancia na captura (proposital, fica pra etapa de NLP).

ITENS_POR_PAGINA_TRATABRASIL = 10


def listar_tratabrasil(html: str, url_base: str, max_paginas: int = 4) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_base if pagina == 1 else f"{url_base.rstrip('/')}/{pagina}/"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        itens_pagina = soup.select("div.e-loop-item")
        if not itens_pagina:
            break

        for item in itens_pagina:
            tag_a = item.select_one("h1.elementor-heading-title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)

            data_publicacao = None
            tag_data = item.select_one('[itemprop="datePublished"] time')
            if tag_data:
                m = re.match(r"(\d{2})/(\d{2})/(\d{4})", tag_data.get_text(strip=True))
                if m:
                    dia, mes, ano = m.groups()
                    data_publicacao = f"{ano}-{mes}-{dia}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": url_item,
                "published_at": data_publicacao,
            })

        if len(itens_pagina) < ITENS_POR_PAGINA_TRATABRASIL:
            break

    return itens

# --- ABRACE Energia (Notícias) ---
#
# WordPress -- loop de posts em div.post-item (titulo/link em
# h5.post-title a, resumo em p.from_the_blog_excerpt, nao usado nos
# metadados). Sem data na listagem -- so na pagina individual, em
# time.entry-date[datetime] (ISO 8601, so fatiar os 10 primeiros
# caracteres). Texto completo reaproveita .entry-content, ja no
# SELETORES_CONTEUDO compartilhado -- sem seletor novo.
#
# Cuidado: a pagina individual tem DOIS <h1> -- o primeiro e o cabecalho
# generico do template ("Noticias", igual em toda pagina do blog), o
# segundo (h1.entry-title) e o titulo de verdade. O extrator generico
# extrair_titulo_h1() pega o primeiro <h1> da pagina -- pegaria "Noticias"
# errado aqui. Por isso "extrair_titulo": None no CONFIGS_FONTES (mantem
# o titulo ja certo, vindo da listagem), mesmo padrao de
# agesan_noticias/agetransp.
#
# Historico medio (40 paginas / ~480 posts) -- max_paginas conservador,
# mesmo criterio de ANP/ABEGAS/ABAR/Trata Brasil.

ITENS_POR_PAGINA_ABRACE = 12


def listar_abrace(html: str, url_base: str, max_paginas: int = 4) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_base if pagina == 1 else f"{url_base.rstrip('/')}/page/{pagina}/"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        itens_pagina = soup.select("div.post-item")
        if not itens_pagina:
            break

        for item in itens_pagina:
            tag_a = item.select_one("h5.post-title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)

            itens.append({"titulo": tag_a.get_text(strip=True), "url": url_item})

        if len(itens_pagina) < ITENS_POR_PAGINA_ABRACE:
            break

    return itens


def data_abrace(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    tag_time = soup.select_one("time.entry-date[datetime]")
    if tag_time and tag_time.get("datetime"):
        return tag_time["datetime"][:10]
    return None


# --- AESBE (Notícias) ---
#
# WordPress + Elementor -- listagem com dois mecanismos de duplicacao
# empilhados. (1) um widget "hero" (os 2 posts mais recentes, em
# article.elementor-post SEM classe ecs-post-loop, titulo em
# h3.elementor-post__title a) que se sobrepoe com a lista paginada de
# verdade (article.elementor-post.ecs-post-loop, titulo em
# h1.elementor-heading-title a) -- mesmo padrao do hero fixo ja visto na
# ABAR. (2) dentro de CADA article.ecs-post-loop, o mesmo titulo/link
# aparece 3x seguidas -- nao sao 3 posts diferentes, sao 3 variantes
# responsivas do mesmo bloco Elementor (desktop/tablet/mobile). Resolvido
# pegando so o primeiro h3/h1 de titulo por article (resolve o 3x
# interno) + dedup por URL entre articles (resolve hero vs. lista).
#
# Sem data na listagem -- so na pagina individual, em
# .elementor-post-info__item--type-date time, como TEXTO PURO
# "DD/MM/YYYY" (sem atributo datetime, diferente de ABRACE/Trata Brasil
# -- aqui precisa de regex mesmo). Texto completo reaproveita
# .elementor-widget-theme-post-content, ja no SELETORES_CONTEUDO
# compartilhado (mesmo seletor do Trata Brasil) -- sem seletor novo.
#
# Pagina individual tem 4 <h1> (o titulo de verdade e o primeiro, os
# outros 3 sao de um widget lateral "Edicao No ..." de newsletter, sem
# relacao com a noticia) -- por seguranca, "extrair_titulo": None no
# CONFIGS_FONTES (mantem o titulo ja certo da listagem), mesmo raciocinio
# do H1 multiplo ja visto em ABRACE.
#
# Historico medio (10 paginas / ~140 posts) -- max_paginas conservador,
# mesmo criterio de ANP/ABEGAS/ABAR/Trata Brasil/ABRACE.

PADRAO_DATA_AESBE = re.compile(r"(\d{2})/(\d{2})/(\d{4})")
ITENS_POR_PAGINA_AESBE = 14


def listar_aesbe(html: str, url_base: str, max_paginas: int = 4) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_base if pagina == 1 else f"{url_base.rstrip('/')}/{pagina}/"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        artigos = soup.select("article.elementor-post")
        if not artigos:
            break

        for artigo in artigos:
            tag_a = artigo.select_one("h3.elementor-post__title a") or artigo.select_one("h1.elementor-heading-title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)

            itens.append({"titulo": tag_a.get_text(strip=True), "url": url_item})

        if len(artigos) < ITENS_POR_PAGINA_AESBE:
            break

    return itens


def data_aesbe(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    tag_data = soup.select_one(".elementor-post-info__item--type-date time")
    if not tag_data:
        return None
    m = PADRAO_DATA_AESBE.search(tag_data.get_text(strip=True))
    if not m:
        return None
    dia, mes, ano = m.groups()
    return f"{ano}-{mes}-{dia}"


# --- TCU (Notícias filtradas por "Solução Consensual") ---
#
# Bloqueio de bot na requisicao simples (challenge JS Akamai, cookie
# TSPD) -- curl_cffi com impersonation de TLS resolve, mesmos headers ja
# usados no resto do projeto. Site Next.js (App Router, RSC streaming),
# mas SSR de verdade -- diferente da ANATEL (onde a listagem era so a
# casca do SearchBlock React), aqui titulo/data/resumo/link ja vem
# prontos no HTML puro.
#
# Paginacao via query string "&pagina=N" (em portugues -- testados varios
# nomes, so esse muda o conteudo), preservando o "tema=". Payload RSC
# embutido no HTML confirma pageSize=15, totalElements=93, totalPages=7.
# Fonte pequena -- max_paginas=7 cobre o arquivo inteiro (diferente de
# ANP/ABEGAS/ABAR, historicos grandes capturados so parcialmente). Sem
# duplicacao de itens (diferente de ABAR/AESBE) -- cada pagina tem
# exatamente os itens esperados.
#
# Data e titulo ja vem prontos e corretos na listagem -- extrair_data e
# extrair_titulo ambos None no CONFIGS_FONTES. Texto completo reaproveita
# extrair_texto_generico() sem nenhuma mudanca -- o fallback pra
# soup.find("article") (quando nenhum seletor de SELETORES_CONTEUDO bate)
# ja da texto limpo aqui.


def listar_tcu_consenso(html: str, url_base: str, max_paginas: int = 7) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = f"{url_base}&pagina={pagina}"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        anchors = soup.select('a[href^="/imprensa/noticias/"]')
        anchors = [a for a in anchors if a.select_one("article")]
        if not anchors:
            break

        novos_na_pagina = 0
        for a in anchors:
            url_item = urllib.parse.urljoin("https://portal.tcu.gov.br", a["href"].strip())
            if url_item in vistos:
                continue
            vistos.add(url_item)
            novos_na_pagina += 1

            h2 = a.select_one("h2")
            titulo = h2.get_text(strip=True) if h2 else None
            if not titulo:
                continue

            data_publicacao = None
            tag_time = a.select_one("time[datetime]")
            if tag_time and tag_time.get("datetime"):
                data_publicacao = tag_time["datetime"][:10]

            itens.append({
                "titulo": titulo,
                "url": url_item,
                "published_at": data_publicacao,
            })

        if not novos_na_pagina:
            break

    return itens

# --- TELETIME News (Notícias) ---
#
# robots.txt bloqueia ClaudeBot nominalmente (junto com GPTBot, CCBot,
# MJ12bot) -- usuario autorizou explicitamente prosseguir mesmo assim
# antes de qualquer requisicao ser feita.
#
# WordPress com tema tagDiv "Newspaper" (nao Elementor/jeg/GT3 como as
# fontes anteriores) -- listagem em div.td_module_wrap, titulo/link em
# h3.entry-title a, data em time.td-module-date (o primeiro item da
# pagina vem com atributo datetime vazio -- fallback por regex no texto
# "DD/MM/AA, HH:MM"). Sem bloqueio tecnico (WAF/anti-bot) -- httpx
# simples ja basta, sem precisar de curl_cffi/impersonation. Sem
# duplicacao -- 36 itens unicos por pagina confirmados.
#
# Texto completo em .td-post-content (acrescentado a SELETORES_CONTEUDO,
# celula de configuracao) -- sem paywall, texto integral confirmado numa
# amostra lida por completo.
#
# Historico gigantesco (1.967 paginas) -- max_paginas conservador, mesmo
# criterio de ABEGAS/ABAR/Trata Brasil.

PADRAO_DATA_TELETIME = re.compile(r"(\d{2})/(\d{2})/(\d{2}),?\s*(\d{2}):(\d{2})")
ITENS_POR_PAGINA_TELETIME = 36


def listar_teletime(html: str, url_base: str, max_paginas: int = 5) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_base if pagina == 1 else f"{url_base.rstrip('/')}/page/{pagina}/"
        html_pagina = html if pagina == 1 else baixar_pagina(url_pagina)
        if not html_pagina:
            break

        soup = BeautifulSoup(html_pagina, "lxml")
        itens_pagina = soup.select("div.td_module_wrap")
        if not itens_pagina:
            break

        for item in itens_pagina:
            tag_a = item.select_one("h3.entry-title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)

            data_publicacao = None
            tag_data = item.select_one("time.td-module-date")
            if tag_data:
                datetime_attr = tag_data.get("datetime") or ""
                if datetime_attr:
                    data_publicacao = datetime_attr[:10]
                else:
                    m = PADRAO_DATA_TELETIME.search(tag_data.get_text(strip=True))
                    if m:
                        dia, mes, ano2, _hh, _mm = m.groups()
                        data_publicacao = f"20{ano2}-{mes}-{dia}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": url_item,
                "published_at": data_publicacao,
            })

        if len(itens_pagina) < ITENS_POR_PAGINA_TELETIME:
            break

    return itens


# --- AGERGS — Notícias ---
#
# CMS "Matriz" (RS) -- a página pública de listagem (/noticias) tem o
# corpo da lista vazio no HTML estático (matriz-ui-pagedlist-body fica em
# branco); os itens são carregados via chamada AJAX em
# /_service/conteudo/pagedlistfilho, que só responde com POST + header
# X-Requested-With: XMLHttpRequest (GET devolve 400 "Serviço
# Indisponível"), e os parâmetros têm que vir exatamente como
# currentPage/pageSize (não pagenumber/pagesize -- nome certo veio do
# próprio erro devolvido pelo endpoint ao tentar o nome errado).
#
# CUIDADO: uma tentativa de GET com querystring nesse mesmo endpoint (em
# vez de POST) causou reset de conexão TLS e o domínio inteiro passou a
# dar timeout por alguns minutos -- usar sempre POST, nunca GET, nesse
# endpoint, e não tentar variantes se der erro.
#
# Por isso listar_agergs() ignora o `html` recebido pelo loop principal
# (que é só a página pública /noticias, usada apenas para o
# baixar_pagina() genérico não falhar) e faz a própria chamada POST.
# Resposta é JSON com um campo "body" contendo fragmentos HTML de cada
# item (título, link e datetime prontos) -- histórico pequeno (9 itens no
# total em 18/08/2026), pageSize alto cobre tudo numa chamada só, sem
# paginação necessária.
#
# extrair_titulo=None de propósito: a página do artigo tem 3 <h1>
# (masthead institucional oculto "text-hide" ANTES do título real
# "artigo__titulo", e depois "Comentários") -- soup.find("h1") pegaria o
# masthead errado, mesmo bug já visto em AESBE/ABRACE.

URL_AGERGS_AJAX = "https://agergs.rs.gov.br/_service/conteudo/pagedlistfilho"
ID_AGERGS_NOTICIAS = "11547"


def listar_agergs(html: str, url_base: str) -> list[dict]:
    try:
        resp = httpx.post(
            URL_AGERGS_AJAX,
            params={"id": ID_AGERGS_NOTICIAS, "templatename": "pagina.listapagina.padrao"},
            data={"ordem": "RECENTES", "currentPage": "1", "pageSize": "50"},
            headers={
                "User-Agent": random.choice(USER_AGENTS),
                "X-Requested-With": "XMLHttpRequest",
                "Accept-Language": "pt-BR,pt;q=0.9",
            },
            timeout=HTTP_TIMEOUT,
        )
        dados = resp.json()
    except Exception:
        return []

    soup = BeautifulSoup(dados.get("body", ""), "lxml")
    itens = []
    for artigo in soup.find_all("article"):
        tag_titulo = artigo.find("h2", class_="conteudo-lista__item__titulo")
        tag_a = tag_titulo.find("a", href=True) if tag_titulo else None
        if not tag_a:
            continue
        titulo = tag_a.get_text(" ", strip=True)
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())

        published_at = None
        tag_time = artigo.find("time")
        if tag_time and tag_time.get("datetime"):
            published_at = tag_time["datetime"][:10]

        itens.append({"titulo": titulo, "url": url_absoluta, "published_at": published_at})
    return itens

# --- ABCR ---
#
# WordPress + Elementor (mesmo widget "eael-grid-post" do Essential
# Addons) -- listagem em article.eael-grid-post, titulo/link em
# h2.eael-entry-title a, resumo em .eael-grid-post-excerpt p (nao usado
# nos metadados). Paginacao real e AJAX-only (botao "Carregar mais",
# admin-ajax.php) -- /noticias/page/N/ NAO e paginacao real (devolve o
# mesmo conteudo da pagina 1, confirmado em Fase 1, teste_abcr.ipynb) --
# captura limitada ao primeiro lote (4 itens), aceito por instrucao
# explicita.
#
# Sem data na listagem -- so na pagina individual, no primeiro <time>
# (texto por extenso no formato "MES DIA, ANO", ex. "agosto 12, 2026" --
# diferente do "DIA de MES de ANO" ja visto em outras fontes, mas
# reaproveita o mesmo dicionario MESES_PT). Texto completo em
# .elementor-widget-theme-post-content, ja no SELETORES_CONTEUDO
# compartilhado (mesmo seletor de Trata Brasil/AESBE) -- sem seletor
# novo. Pagina individual tem 2 <h1> mas ambos com o titulo correto
# (sem masthead escondido como AESBE/ABRACE) -- extrair_titulo_h1
# generico funciona normalmente.
#
# robots.txt tem bloco Cloudflare que nomeia varios bots de IA
# (ClaudeBot, GPTBot, Google-Extended etc.) com Disallow: /, mas o grupo
# generico User-agent: * permite tudo (Allow: /) -- usuario autorizou
# explicitamente prosseguir mesmo assim antes de qualquer requisicao ser
# feita, mesmo padrao ja visto em TELETIME.

PADRAO_DATA_ABCR = re.compile(r"(\w+)\s+(\d{1,2}),\s*(\d{4})")


def listar_abcr(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens = []
    for artigo in soup.find_all("article", class_="eael-grid-post"):
        tag_a = artigo.select_one("h2.eael-entry-title a")
        if not tag_a:
            continue
        titulo = tag_a.get_text(" ", strip=True)
        url_absoluta = urllib.parse.urljoin(url_base, tag_a["href"].strip())
        itens.append({"titulo": titulo, "url": url_absoluta})
    return itens


def data_abcr(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    tag_time = soup.find("time")
    if not tag_time:
        return None
    m = PADRAO_DATA_ABCR.search(tag_time.get_text(strip=True))
    if not m:
        return None
    mes_nome, dia, ano = m.groups()
    mes = MESES_PT.get(mes_nome.lower())
    if not mes:
        return None
    return f"{ano}-{mes}-{dia.zfill(2)}"


In [0]:
CONFIGS_FONTES = {
    "acende_brasil": {
        "site_url": "https://acendebrasil.com.br/artigos/",
        "source_id": "acende_brasil",
        "tema": "ENERGIA",
        "source_descricao": "Linked from Instituto Acende Brasil — Artigos",
        "listar": listar_acende_brasil,
        "extrair_data": data_acende_brasil,
        "extrair_titulo": extrair_titulo_h1,
    },
    "antt": {
        "site_url": (
            "https://www.gov.br/antt/pt-br/assuntos/noticias-defeso-eleitoral"
            "?form.submitted=1&texto=&dt_inicio=&dt_fim=&categoria=infraestrutura-transito-e-transportes"
        ),
        "source_id": "antt_noticias_infraestrutura",
        "tema": "TRANSPORTE",
        "source_descricao": "Linked from ANTT — Notícias (Infraestrutura, Trânsito e Transportes)",
        "listar": listar_antt,
        "extrair_data": data_antt,
        "extrair_titulo": extrair_titulo_h1,
    },
    "agesan_noticias": {
        "site_url": "https://agesan-rs.com.br/noticias/",
        "source_id": "agesan_rs_noticias",
        "tema": "SANEAMENTO",
        "source_descricao": "Linked from Agesan-RS — Notícias",
        "listar": listar_agesan_noticias,
        "extrair_data": None,
        "extrair_titulo": None,
    },
    "psr_exame": {
        "site_url": "https://exame.com/arquivo/psr-energia-em-foco/",
        "source_id": "psr_energia_em_foco",
        "tema": "ENERGIA",
        "source_descricao": "Linked from Exame — PSR Energia em Foco",
        "listar": listar_psr_exame,
        "extrair_data": data_psr_exame,
        "extrair_titulo": extrair_titulo_h1_exame,
        "requer_data": True,
    },
    "aneel": {
        "site_url": "https://www.gov.br/aneel/pt-br/assuntos/noticias",
        "source_id": "aneel_noticias",
        "tema": "ENERGIA",
        "source_descricao": "Linked from ANEEL — Notícias",
        "listar": lambda html, url: listar_govbr_plone(
            html, url, "https://www.gov.br/aneel/pt-br/assuntos/noticias/2026-defeso-eleitoral"
        ),
        "extrair_data": data_govbr_plone,
        "extrair_titulo": extrair_titulo_h1,
    },
    "agetransp": {
        "site_url": "https://www.agetransp.rj.gov.br/noticias",
        "source_id": "agetransp_noticias",
        "tema": "TRANSPORTE",
        "source_descricao": "Linked from AGETRANSP — Notícias",
        "listar": listar_agetransp,
        "extrair_data": None,
        "extrair_titulo": None,
    },
    "ana": {
        # TODO (pós-período eleitoral 2026): endereço temporário -- ver
        # comentário completo junto de listar_ana(), na célula acima.
        "site_url": "https://www.gov.br/ana/pt-br/assuntos/noticias-e-eventos/noticias-periodo-eleitoral-2026/",
        "source_id": "ana_noticias",
        "tema": "SANEAMENTO",
        "source_descricao": "Linked from ANA — Notícias (Agência Nacional de Águas e Saneamento Básico)",
        "listar": lambda html, url: listar_ana(html, url, max_paginas=5),
        "extrair_data": data_govbr_plone,
        "extrair_titulo": extrair_titulo_h1,
    },
    "agenersa": {
        "site_url": "https://www.rj.gov.br/agenersa/noticias",
        "source_id": "agenersa_rj_noticias",
        "tema": "SANEAMENTO",
        "source_descricao": "Linked from AGENERSA (RJ) — Notícias",
        "listar": lambda html, url: listar_agenersa(html, url, max_paginas=5),
        "extrair_data": None,
        "extrair_titulo": extrair_titulo_h1,
    },
    "anp": {
        # Mesma plataforma Plone/gov.br da ANA -- listar_ana() é genérica
        # (não tem nada específico da ANA apesar do nome), reaproveitada
        # aqui sem alteração. Sem URL temporária (endereço permanente).
        "site_url": "https://www.gov.br/anp/pt-br/canais_atendimento/imprensa/noticias-comunicados",
        "source_id": "anp_noticias",
        "tema": "ENERGIA",
        "source_descricao": "Linked from ANP — Notícias e Comunicados (Agência Nacional do Petróleo, Gás Natural e Biocombustíveis)",
        "listar": lambda html, url: listar_ana(html, url, max_paginas=5),
        "extrair_data": data_govbr_plone,
        "extrair_titulo": extrair_titulo_h1,
    },
    "abegas": {
        # WordPress com tema page-builder proprio (GT3/Elementor) -- ver
        # comentario completo junto de listar_abegas(), na celula acima.
        "site_url": "https://www.abegas.org.br/noticias-do-setor",
        "source_id": "abegas_noticias",
        "tema": "ENERGIA",
        "source_descricao": "Linked from ABEGÁS — Notícias do Setor (Associação Brasileira das Empresas Distribuidoras de Gás Canalizado)",
        "listar": lambda html, url: listar_abegas(html, url, max_paginas=5),
        "extrair_data": None,
        "extrair_titulo": extrair_titulo_h1,
    },
    "anatel": {
        # Plone 6/Volto -- ver comentario completo junto de listar_anatel(),
        # na celula acima. site_url e a URL da API de busca, nao a pagina
        # publica (a pagina de listagem e so a casca do SearchBlock React).
        "site_url": (
            "https://www.gov.br/anatel/++api++/@search"
            "?portal_type=News%20Item&path=/pt-br/assuntos/noticias"
            "&sort_on=effective&sort_order=descending&b_size=100"
        ),
        "source_id": "anatel_noticias",
        "tema": "TELECOM",
        "source_descricao": "Linked from ANATEL — Notícias (Agência Nacional de Telecomunicações)",
        "listar": listar_anatel,
        "extrair_data": None,
        "extrair_titulo": extrair_titulo_h1,
    },
    "abar": {
        # Agregador nacional (87 agências associadas) -- tema "REGULATORIO"
        # em vez de um setor único, já que o conteúdo mistura saneamento,
        # energia, transporte e recursos hídricos por natureza. Ver
        # comentário completo junto de listar_abar(), na célula acima.
        "site_url": "https://abar.org.br/category/acontece-nas-agencias/",
        "source_id": "abar_noticias",
        "tema": "REGULATORIO",
        "source_descricao": "Linked from ABAR — Acontece nas Agências (Associação Brasileira de Agências Reguladoras)",
        "listar": lambda html, url: listar_abar(html, url, max_paginas=5),
        "extrair_data": None,
        "extrair_titulo": extrair_titulo_h1,
    },
    "epe": {
    "site_url": "https://www.epe.gov.br/pt/imprensa/noticias",
    "source_id": "epe_noticias",
    "source_descricao": "Linked from EPE — Notícias",
    "listar": listar_epe,
    "extrair_data": None,
    "extrair_titulo": None,
},
    "tratabrasil": {
        # WordPress + Elementor -- ver comentario completo junto de
        # listar_tratabrasil(), na celula acima.
        "site_url": "https://tratabrasil.org.br/blog/",
        "source_id": "tratabrasil_blog",
        "tema": "SANEAMENTO",
        "source_descricao": "Linked from Instituto Trata Brasil — Blog",
        "listar": lambda html, url: listar_tratabrasil(html, url, max_paginas=4),
        "extrair_data": None,
        "extrair_titulo": extrair_titulo_h1,
    },
    "antaq": {
    "site_url": "https://www.gov.br/antaq/pt-br/noticias",
    "source_id": "antaq_noticias",
    "tema": "TRANSPORTE",
    "source_descricao": "Linked from ANTAQ — Notícias (Agência Nacional de Transportes Aquaviários)",
    "listar": lambda html, url: listar_govbr_plone(
        html, url, "https://www.gov.br/antaq/pt-br/noticias"
    ),
    "extrair_data": data_govbr_plone,
    "extrair_titulo": extrair_titulo_h1,
},
    "abrace": {
        # WordPress -- ver comentario completo junto de listar_abrace(),
        # na celula acima. extrair_titulo=None de proposito (ver o bug do
        # H1 duplo documentado la).
        "site_url": "https://abrace.org.br/noticias/",
        "source_id": "abrace_noticias",
        "tema": "ENERGIA",
        "source_descricao": "Linked from ABRACE Energia — Notícias (Associação Brasileira de Grandes Consumidores Industriais de Energia)",
        "listar": lambda html, url: listar_abrace(html, url, max_paginas=4),
        "extrair_data": data_abrace,
        "extrair_titulo": None,
    },
    "aesbe": {
        # WordPress + Elementor -- ver comentario completo junto de
        # listar_aesbe(), na celula acima. extrair_titulo=None de
        # proposito (pagina tem 4 h1, so o primeiro e o titulo de
        # verdade -- por seguranca mantem o titulo ja certo da listagem).
        "site_url": "https://aesbe.org.br/noticias-externa/",
        "source_id": "aesbe_noticias",
        "tema": "SANEAMENTO",
        "source_descricao": "Linked from AESBE — Notícias (Associação Brasileira das Empresas Estaduais de Saneamento)",
        "listar": lambda html, url: listar_aesbe(html, url, max_paginas=4),
        "extrair_data": data_aesbe,
        "extrair_titulo": None,
    },
    "tcu_consenso": {
        # Ver comentario completo junto de listar_tcu_consenso(), na
        # celula acima. Data e titulo ja vem prontos na listagem.
        "site_url": "https://portal.tcu.gov.br/imprensa/noticias?tema=" + urllib.parse.quote("Solução consensual"),
        "source_id": "tcu_solucao_consensual",
        "tema": "REGULATORIO",
        "source_descricao": "Linked from TCU — Notícias (Solução Consensual)",
        "listar": lambda html, url: listar_tcu_consenso(html, url, max_paginas=7),
        "extrair_data": None,
        "extrair_titulo": None,
    },
    "teletime": {
        # Ver comentario completo junto de listar_teletime(), na celula
        # acima. Data ja vem pronta na listagem.
        "site_url": "https://teletime.com.br/noticias/",
        "source_id": "teletime_noticias",
        "tema": "TELECOM",
        "source_descricao": "Linked from TELETIME News — Notícias",
        "listar": lambda html, url: listar_teletime(html, url, max_paginas=5),
        "extrair_data": None,
        "extrair_titulo": extrair_titulo_h1,
    },
    "agergs": {
        # CMS Matriz -- ver comentario completo junto de listar_agergs(),
        # na celula acima. site_url e a pagina publica /noticias, so para
        # o baixar_pagina() generico nao falhar -- listar_agergs() ignora
        # esse html e faz a propria chamada POST ao endpoint AJAX.
        "site_url": "https://agergs.rs.gov.br/noticias",
        "source_id": "agergs_noticias",
        "tema": "TRANSPORTE",
        "source_descricao": "Linked from AGERGS — Notícias (Agência Estadual de Regulação dos Serviços Públicos Delegados do Rio Grande do Sul)",
        "listar": listar_agergs,
        "extrair_data": None,
        "extrair_titulo": None,
    },
    "abcr": {
        # WordPress + Elementor -- ver comentario completo junto de
        # listar_abcr(), na celula acima. Captura limitada ao primeiro
        # lote (4 itens) -- paginacao real e AJAX-only, nao implementada
        # por instrucao explicita.
        "site_url": "https://melhoresrodovias.org.br/noticias/",
        "source_id": "abcr",
        "tema": "TRANSPORTE",
        "source_descricao": "Linked from ABCR — Notícias (Associação Brasileira de Concessionárias de Rodovias)",
        "listar": listar_abcr,
        "extrair_data": data_abcr,
        "extrair_titulo": extrair_titulo_h1,
    },
}

In [0]:
def salvar_artefatos(pasta: str, source_id: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(source_id, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


def processar_item(item: dict, config: dict, pasta_destino: str) -> Optional[dict]:
    titulo_listagem = item["titulo"]
    url = item["url"]
    print(f"\n  [item] {titulo_listagem[:100]}")

    html = baixar_pagina(url)
    if not html:
        print("    -> download falhou; pulando.")
        return None

    titulo = titulo_listagem
    if config["extrair_titulo"]:
        titulo = config["extrair_titulo"](html) or titulo_listagem

    texto = extrair_texto_generico(html)
    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> texto muito curto ({len(texto)} chars); pulando.")
        return None

    data_publicacao = item.get("published_at")
    if config["extrair_data"]:
        data_publicacao = config["extrair_data"](html)

    if not data_publicacao:
        data_publicacao = HOJE

    if config.get("requer_data") and not data_publicacao:
        print("    -> sem data de publicação identificável; provável página de ferramenta/índice, pulando.")
        return None

    metadados = {
        "source_id": config["source_id"],
        "title": titulo,
        "description": config["source_descricao"],
        "url": url,
        "date": HOJE,
        "published_at": data_publicacao,
    }

    caminho_txt, caminho_json = salvar_artefatos(pasta_destino, config["source_id"], titulo, texto, metadados)
    print(f"    -> salvo em {caminho_txt}")
    return {"titulo": titulo, "url": url, "caminho_txt": caminho_txt, "caminho_json": caminho_json}

In [0]:
if NOME_FONTE == "todas":
    fontes_a_rodar = CONFIGS_FONTES
else:
    if NOME_FONTE not in CONFIGS_FONTES:
        raise ValueError(f"Fonte {NOME_FONTE!r} não configurada. Opções: {list(CONFIGS_FONTES)}")
    fontes_a_rodar = {NOME_FONTE: CONFIGS_FONTES[NOME_FONTE]}

resumo_geral = {}

for nome_fonte, config in fontes_a_rodar.items():
    print(f"\n{'='*70}\n=== Fonte: {nome_fonte!r} ===\n{'='*70}")

    caminho_manifesto = os.path.join(PASTA_MANIFESTOS, f"{config['source_id']}_processados.json")
    ja_processados = carregar_manifesto(caminho_manifesto)

    pasta_destino_fonte = PASTA_BASE
    os.makedirs(pasta_destino_fonte, exist_ok=True)

    try:
        html_listagem = baixar_pagina(config["site_url"])
        if not html_listagem:
            resumo_geral[nome_fonte] = "ERRO: download da listagem falhou"
            atualizar_status_fonte(
                source_id=config["source_id"],
                sucesso=False,
                docs_capturados=0,
                erro="download da listagem falhou",
            )
            continue

        itens_pagina = config["listar"](html_listagem, config["site_url"])
        itens_novos = [i for i in itens_pagina if i["url"] not in ja_processados]
        print(f"{len(itens_pagina)} itens na página, {len(itens_novos)} novos.")

        salvos = 0
        for item in itens_novos:
            try:
                resultado = processar_item(item, config, pasta_destino_fonte)
                if resultado:
                    salvos += 1
                    ja_processados.add(item["url"])
            except Exception as e:
                print(f"[ERRO] item {item['titulo']!r} falhou: {e}")

        salvar_manifesto(caminho_manifesto, ja_processados)
        resumo_geral[nome_fonte] = f"{salvos} novos salvos"

        atualizar_status_fonte(
            source_id=config["source_id"],
            sucesso=True,
            docs_capturados=salvos,
        )

    except Exception as e:
        resumo_geral[nome_fonte] = f"ERRO: {e}"

        atualizar_status_fonte(
            source_id=config["source_id"],
            sucesso=False,
            docs_capturados=0,
            erro=str(e),
        )

print(f"\n\n{'='*70}\n=== RESUMO ===")
for nome_fonte, resultado in resumo_geral.items():
    print(f"  {nome_fonte}: {resultado}")
